# Regression Model Comparison

The model-specific experiments now live in five separate notebooks:

| Notebook | Model |
|---|---|
| `03a_linear_regression.ipynb` | Linear Regression |
| `03b_ridge_regression.ipynb` | Ridge Regression |
| `03c_decision_tree.ipynb` | Decision Tree |
| `03d_random_forest.ipynb` | Random Forest |
| `03e_gradient_boosting.ipynb` | Gradient Boosting |

Each notebook includes initial modelling, selection/tuning, fold scores, training/validation diagnostics, and saved pipelines for both feature scenarios. Run those notebooks before this one.

This notebook verifies the shared population, split, folds, code/data fingerprints, and artifact fingerprints before comparing results. It ranks models separately for the no-prior-grades and G1/G2-available scenarios. The latter makes a prediction at a later time and cannot substitute for an earlier-support scenario.

MAE is the primary selection metric. All reported scores are training cross-validation selection scores. Final held-out evaluation remains Member 4's responsibility. The earlier combined notebook is retained as a historical snapshot in `archive/03_regression_modelling_combined_2026-10-11.ipynb`.

## Load and Validate Completed Experiments

In [1]:
from pathlib import Path
import hashlib
import json
import shutil
import sys

import joblib
import pandas as pd

PROJECT_ROOT = next(
    folder for folder in [Path.cwd(), *Path.cwd().parents]
    if (folder / "data" / "student-mat.csv").is_file()
    and (folder / "src" / "preprocessing.py").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from preprocessing import make_preprocessor

slugs = ["linear_regression", "ridge", "decision_tree", "random_forest", "gradient_boosting"]
experiments = {}
for slug in slugs:
    path = PROJECT_ROOT / "docs" / "experiments" / "per_model" / f"{slug}.json"
    if not path.is_file():
        raise FileNotFoundError(f"Run the {slug} notebook first: {path}")
    experiments[slug] = json.loads(path.read_text(encoding="utf-8"))

reference = experiments[slugs[0]]
current_data_hash = hashlib.sha256((PROJECT_ROOT / "data" / "student-mat.csv").read_bytes()).hexdigest()
current_code_hash = hashlib.sha256((PROJECT_ROOT / "src" / "preprocessing.py").read_bytes()).hexdigest()

for slug, record in experiments.items():
    for key in ["students", "zero_grades", "training_indices", "test_indices", "fold_validation_indices", "dataset_sha256", "preprocessing_sha256", "versions"]:
        assert record[key] == reference[key], f"Inconsistent experiment: {slug}, {key}"
    assert record["dataset_sha256"] == current_data_hash, f"Dataset changed; rerun {slug}."
    assert record["preprocessing_sha256"] == current_code_hash, f"Preprocessing changed; rerun {slug}."
    assert record["students"] == 395 and record["zero_grades"] == 38
    assert record["test_performance_evaluated"] is False
    assert set(record["scenarios"]) == {"no_prior_grades", "with_prior_grades"}
    for scenario, result in record["scenarios"].items():
        assert result["include_prior_grades"] == (scenario == "with_prior_grades")
        assert (set(["G1", "G2"]) <= set(result["input_features"])) == result["include_prior_grades"]
        artifact = PROJECT_ROOT / result["pipeline_path"]
        assert hashlib.sha256(artifact.read_bytes()).hexdigest() == result["pipeline_sha256"], f"Artifact changed: {artifact}"

assert len(reference["training_indices"]) == 316
assert len(reference["test_indices"]) == 79
assert not set(reference["training_indices"]) & set(reference["test_indices"])
print("All five experiments use identical students, folds, data, code, and package versions.")

All five experiments use identical students, folds, data, code, and package versions.


## Compare Within Each Prediction Scenario

In [2]:
rows = []
for slug, record in experiments.items():
    for scenario, result in record["scenarios"].items():
        rows.append({
            "Scenario": scenario,
            "Model": record["model"],
            "Initial MAE": result["initial_scores"]["MAE"],
            "Selected MAE": result["selected_scores"]["MAE"],
            "Selected RMSE": result["selected_scores"]["RMSE"],
            "Selected R2": result["selected_scores"]["R2"],
            "MAE fold std": result["selected_std"]["MAE"],
            "Log absences": result["log_absences"],
            "slug": slug
        })
comparison = pd.DataFrame(rows)
scenario_rankings = {}
for scenario in ["no_prior_grades", "with_prior_grades"]:
    ranking = comparison.loc[comparison["Scenario"] == scenario].sort_values("Selected MAE").reset_index(drop=True)
    scenario_rankings[scenario] = ranking
    print("Prediction scenario:", scenario)
    display(ranking.drop(columns=["Scenario", "slug"]))

first_scenario = reference["scenarios"]["no_prior_grades"]
print("Shared constant baselines:")
display(pd.DataFrame({
    "Mean baseline": first_scenario["mean_baseline_scores"],
    "Median baseline": first_scenario["median_baseline_scores"]
}).T)

Prediction scenario: no_prior_grades


,Model,Initial MAE,Selected MAE,Selected RMSE,Selected R2,MAE fold std,Log absences
0,Random Forest,3.001524,2.964054,3.936634,0.208057,0.345624,False
1,Gradient Boosting,3.046055,3.046055,3.945861,0.210174,0.409967,False
2,Decision Tree,3.130386,3.079132,4.063580,0.154912,0.323814,False
3,Ridge Regression,3.490391,3.242256,4.167685,0.125253,0.445422,True
4,Linear Regression,3.522677,3.476333,4.461546,-0.030799,0.345854,True


Prediction scenario: with_prior_grades


,Model,Initial MAE,Selected MAE,Selected RMSE,Selected R2,MAE fold std,Log absences
0,Random Forest,0.984604,0.938128,1.475705,0.892770,0.205599,True
1,Gradient Boosting,0.984876,0.984876,1.579993,0.876503,0.209574,False
2,Decision Tree,1.219749,1.010811,1.661768,0.864580,0.260109,False
3,Ridge Regression,1.370474,1.295472,1.811573,0.835833,0.206809,True
4,Linear Regression,1.392734,1.372367,1.882924,0.823889,0.221451,True


Shared constant baselines:


,MAE,RMSE,R2
Mean baseline,3.402865,4.563458,-0.035023
Median baseline,3.340278,4.593095,-0.051899


## Compare Available Information Within Each Model

The student rows and validation folds match. Differences therefore reflect the evaluated feature scenario and selected settings. Higher scores with G1/G2 would indicate the value of later-period information, not earlier availability or proven causal effects.

In [3]:
information_comparison = comparison.pivot(
    index="Model", columns="Scenario", values=["Selected MAE", "Selected RMSE", "Selected R2"]
)
information_comparison

Selected MAE  ...       Selected R2
Scenario          no_prior_grades  ... with_prior_grades
Model                              ...                  
Decision Tree            3.079132  ...          0.864580
Gradient Boosting        3.046055  ...          0.876503
Linear Regression        3.476333  ...          0.823889
Random Forest            2.964054  ...          0.892770
Ridge Regression         3.242256  ...          0.835833

[5 rows x 6 columns]

## Save Selected Pipelines for Member 4

Select the lowest-MAE candidate separately within each scenario. Preserve the no-prior-grades scenario as the main project model. The later-stage pipeline is saved separately. No test predictions or test scores are used to choose either model.

In [4]:
selected_records = {}
for scenario, ranking in scenario_rankings.items():
    selected_slug = ranking.iloc[0]["slug"]
    record = experiments[selected_slug]
    result = record["scenarios"][scenario]
    source_path = PROJECT_ROOT / result["pipeline_path"]
    filename = (
        "best_regression_pipeline.joblib"
        if scenario == "no_prior_grades"
        else "best_with_prior_grades_pipeline.joblib"
    )
    selected_path = PROJECT_ROOT / "models" / filename
    shutil.copy2(source_path, selected_path)
    # Confirm that the selected fitted pipeline can be loaded independently.
    loaded_pipeline = joblib.load(selected_path)
    assert loaded_pipeline.named_steps["preprocessing"].named_steps["absence_transform"].kw_args["use_log"] == result["log_absences"]

    selected_record = {
        "model": record["model"],
        "scenario": scenario,
        "students": record["students"],
        "zero_grades": record["zero_grades"],
        "training_indices": record["training_indices"],
        "test_indices": record["test_indices"],
        "fold_validation_indices": record["fold_validation_indices"],
        "split_random_state": 42,
        "test_size": 0.2,
        "selection_metric": "MAE",
        "selection_cv_scores": result["selected_scores"],
        "selection_cv_std": result["selected_std"],
        "model_parameters": result["selected_model_parameters"],
        "input_features": result["input_features"],
        "include_prior_grades": result["include_prior_grades"],
        "log_absences": result["log_absences"],
        "scaling": result["scaling"],
        "dataset_sha256": record["dataset_sha256"],
        "preprocessing_sha256": record["preprocessing_sha256"],
        "versions": record["versions"],
        "source_pipeline_path": result["pipeline_path"],
        "pipeline_path": selected_path.relative_to(PROJECT_ROOT).as_posix(),
        "pipeline_sha256": hashlib.sha256(selected_path.read_bytes()).hexdigest(),
        "test_performance_evaluated": False
    }
    metadata_path = selected_path.with_name(selected_path.stem.replace("_pipeline", "_metadata") + ".json")
    metadata_path.write_text(json.dumps(selected_record, indent=2), encoding="utf-8")
    selected_records[scenario] = selected_record
    print(scenario, "->", record["model"], "| CV MAE:", result["selected_scores"]["MAE"])
    print("Saved:", selected_path.relative_to(PROJECT_ROOT))

no_prior_grades -> Random Forest | CV MAE: 2.9640541013939945
Saved: models\best_regression_pipeline.joblib
with_prior_grades -> Random Forest | CV MAE: 0.9381277479717335
Saved: models\best_with_prior_grades_pipeline.joblib


## Record the Updated Main Analysis

In [5]:
main_results = {
    "scope": "All recorded Mathematics students, including zero final grades; no prior grades in the main scenario",
    "students": reference["students"],
    "zero_grades": reference["zero_grades"],
    "training_students": len(reference["training_indices"]),
    "test_students": len(reference["test_indices"]),
    "training_indices": reference["training_indices"],
    "test_indices": reference["test_indices"],
    "split_random_state": 42,
    "test_size": 0.2,
    "cv": "5-fold shuffled KFold, random_state=42",
    "selection_metric": "MAE",
    "dataset_sha256": current_data_hash,
    "preprocessing_sha256": current_code_hash,
    "versions": reference["versions"],
    "initial_comparison": {record["model"]: record["scenarios"]["no_prior_grades"]["initial_scores"] for record in experiments.values()},
    "tuned_comparison": {record["model"]: record["scenarios"]["no_prior_grades"]["selected_scores"] for record in experiments.values()},
    "selected_model": selected_records["no_prior_grades"]["model"],
    "selected_settings": {record["model"]: {
        "model_parameters": record["scenarios"]["no_prior_grades"]["selected_model_parameters"],
        "log_absences": record["scenarios"]["no_prior_grades"]["log_absences"]
    } for record in experiments.values()},
    "later_stage_selected_model": selected_records["with_prior_grades"]["model"],
    "test_performance_evaluated": False
}
summary_path = PROJECT_ROOT / "docs" / "experiments" / "all_students_results.json"
summary_path.write_text(json.dumps(main_results, indent=2), encoding="utf-8")
print("Saved central results. Member 4 can evaluate the saved pipelines using their recorded test row indices.")

Saved central results. Member 4 can evaluate the saved pipelines using their recorded test row indices.


## Interpretation and Limitations

Do not infer a statistically significant difference from small gaps in mean cross-validation scores. The searches and model comparisons reuse the same folds, so these are selection estimates. The reserved test set remains necessary for final evaluation. G1/G2 must be available at the declared prediction time; their strong correlation with G3 alone does not make them leakage when predicting after the second period. Other feature timing, including absences, still needs review. Zero-grade causes remain unknown.